# Day 27 Solutions: Geospatial and Domain-Specific Visualization

## Lesson walkthrough

GeoPandas world map or NetworkX pipeline graph.

The walkthrough cells run the lesson's original examples. The exercise answers after them contain the same code and reasoning as `day27_solutions.md`.


In [ ]:
# GeoPandas track (requires geopandas, contextily, geodatasets)
# import contextily as cx
# import geopandas as gpd
# from geodatasets import get_path
# world = gpd.read_file(get_path('naturalearth.land')).to_crs(3857)
# ax = world.plot(figsize=(8,6), alpha=0.6, edgecolor='k')
# cx.add_basemap(ax, source=cx.providers.OpenStreetMap.Mapnik)
# ax.set(title='World basemap (Web Mercator)')

In [ ]:
# Domain viz track: network graph of pipeline stages
import networkx as nx, matplotlib.pyplot as plt

G = nx.Graph()
G.add_edge("ingest", "clean")
G.add_edge("clean", "validate")
G.add_edge("validate", "model")
G.add_edge("model", "serve")
G.add_edge("serve", "monitor")
pos = nx.spring_layout(G, seed=42)
plt.figure(figsize=(6, 4))
nx.draw(G, pos, with_labels=True, node_color="lightblue", node_size=1200)
plt.title("Pipeline Stages (Network Graph)")
plt.tight_layout()
plt.show()

<!-- BEGIN BEGINNER SOLUTION REVIEW -->
## Concept review before comparing answers

These worked answers demonstrate **geospatial semantics, coordinate reference systems, valid geometry, and offline fallback**. Predict each named
result before comparing your attempt with its matching assertions.

Geospatial data combines ordinary attributes with geometry and a
coordinate reference system (CRS). A coordinate pair is conventionally
`(x, y)`, longitude then latitude for geographic coordinates. Values can
look numerically valid while being semantically swapped, so record
source order and bounds.

A CRS defines what coordinates mean and their units. Layers cannot be
overlaid or spatially joined honestly until their CRSs are known and
compatible. Reprojection transforms coordinates; assigning a CRS merely
labels existing coordinates and is not a substitute. Inspect validity,
emptiness, bounds, provenance, and offline availability before plotting.

### Vocabulary used in the worked answers

- **geometry:** a spatial object such as point, line, or polygon.
- **CRS:** coordinate reference system defining coordinate meaning and units.
- **reprojection:** transforming geometry coordinates from one CRS to another.
- **longitude:** east-west angular coordinate, used as x in geographic pairs.
- **latitude:** north-south angular coordinate, used as y in geographic pairs.
- **spatial join:** matching records by a geometric relationship such as within or intersects.

### How to compare an answer

For this lesson's **geospatial semantics, coordinate reference systems, valid geometry, and offline fallback** model, follow the exact values from each learner contract through its function or expression to the assertion that proves the expected behavior; then change one boundary input and make that assertion fail once before accepting the answer.
<!-- END BEGINNER SOLUTION REVIEW -->

## Exercises 1-2: Worked answers

### Exercise 1: worked answer

**Learner contract:** Using only a local or already-cached boundary/point source, create a GeoDataFrame plot. **Before plotting:** record provenance/license, CRS, geometry types, invalid/empty counts, and bounds. **Constraints:** transform all layers to one appropriate CRS and never use `set_crs` as if it reprojected coordinates. **Verify:** check a known location/bounds and confirm entity count survives reprojection.

**Reasoning:** Profiling provenance, license, CRS, geometry types, validity, and bounds before plotting catches the common spatial errors early: an unknown CRS, swapped coordinates, or broken shapes. The known-location check confirms that San Francisco's x value is -122.4, which holds only if the points were built in `(longitude, latitude)` order. `to_crs("EPSG:3857")` computes new coordinates in meters for Web Mercator, so the geometries change while the number of entities stays the same; `set_crs` would only relabel the existing numbers, which is correct only when the original CRS label was missing or wrong.


In [ ]:
import geopandas as gpd
from shapely.geometry import Point

places = gpd.GeoDataFrame(
    {"name": ["San Francisco", "New York"]},
    geometry=[Point(-122.4, 37.8), Point(-73.9, 40.7)],
    crs="EPSG:4326",
)
spatial_profile = {
    "provenance": "constructed course points",
    "license": "course-authored fixture; unrestricted educational reuse",
    "crs": places.crs.to_string(),
    "geometry_types": sorted(places.geom_type.unique().tolist()),
    "invalid_count": int((~places.geometry.is_valid).sum()),
    "empty_count": int(places.geometry.is_empty.sum()),
    "bounds": places.total_bounds.tolist(),
}
assert places.crs is not None
assert places.geometry.is_valid.all()
assert spatial_profile["geometry_types"] == ["Point"]
assert spatial_profile["bounds"][0] < -122
assert places.loc[places["name"].eq("San Francisco"), "geometry"].iloc[0].x == -122.4
projected = places.to_crs("EPSG:3857")
assert len(projected) == len(places)
assert projected.crs == "EPSG:3857"
assert not projected.geometry.equals(places.geometry)
ax = projected.plot()
assert len(ax.collections) >= 1

This constructed local fixture needs no basemap or network source.
`to_crs` transforms coordinates; `set_crs` would only relabel their
meaning and would be incorrect here.

### Exercise 2: worked answer

**Learner contract:** Alternatively, construct a deterministic NetworkX domain graph where node and edge meanings are written explicitly. **Constraints:** use a fixed layout seed, encode at most a few meaningful attributes, provide labels/legend, and do not require network data. **Verify:** reconcile plotted node/edge counts to the graph and explain what spatial questions this fallback cannot answer.

**Reasoning:** When geographic data is unavailable, a small graph can still show structure, here which step feeds which. The fixed `seed=42` makes `spring_layout` produce the same positions on every run, node shapes plus a legend encode each node's role without relying on color, and a label on each node names it directly. The reconciliation checks that the drawing shows three nodes and that the graph has two edges. The positions are chosen for readability, so distances in this drawing mean nothing, which is the limitation to state.


In [ ]:
import matplotlib.pyplot as plt
import networkx as nx

graph = nx.Graph()
graph.add_nodes_from(
    [
        ("source", {"role": "input"}),
        ("clean", {"role": "transform"}),
        ("report", {"role": "output"}),
    ]
)
graph.add_edges_from(
    [
        ("source", "clean", {"meaning": "feeds"}),
        ("clean", "report", {"meaning": "produces"}),
    ]
)
positions = nx.spring_layout(graph, seed=42)
fig, ax = plt.subplots(figsize=(5, 3))
role_shapes = {"input": "s", "transform": "o", "output": "^"}
for role, shape in role_shapes.items():
    nodes = [node for node, attributes in graph.nodes(data=True) if attributes["role"] == role]
    nx.draw_networkx_nodes(
        graph,
        positions,
        nodelist=nodes,
        node_shape=shape,
        label=role,
        ax=ax,
    )
nx.draw_networkx_edges(graph, positions, ax=ax)
nx.draw_networkx_labels(graph, positions, ax=ax)
ax.legend(title="Node role")
ax.set_axis_off()

assert graph.number_of_nodes() == 3
assert graph.number_of_edges() == 2
assert set(positions) == set(graph)
assert sum(len(collection.get_offsets()) for collection in ax.collections[:3]) == 3
plt.close(fig)

A deterministic graph can show dependency relationships offline, but
it cannot answer geographic distance, containment, or coordinate
questions because its layout encodes readability, not physical space.

## Exercises 3-7: Expanded mastery answers

### Exercise 3: answer contract

**Learner contract:** **Prediction:** Predict what goes wrong when latitude/longitude degrees are overlaid on a web map measured in Web Mercator meters. **Progressive hint:** Layers must use compatible coordinate reference systems (CRSs). **Verify:** Compare both CRS/bounds, then assert reprojection makes units compatible and the transformed layers overlap in a plausible extent.

**Reasoning:** EPSG:4326 measures in degrees and Web Mercator, EPSG:3857, in meters, so drawing raw longitudes such as -122 on a meter-based map places the points within about 130 meters of the map origin, off the coast of West Africa, far from the United States. The shared implementation for Exercises 3-4 prints both units and bounds, shows that the raw degree box does not lie inside the meter-based extent, and then shows that after `to_crs` both layers share one CRS and the points fall inside the map extent.

### Exercise 4: answer contract

**Learner contract:** **Tracing:** Trace one point represented as `(longitude, latitude)` and explain why swapping values can still create a valid but wrong location. **Progressive hint:** Both numbers may fall in legal ranges, so semantic order matters. **Verify:** Validate the known longitude/latitude pair against real bounds, swap it, and show why semantic checks, not only numeric ranges, detect the wrong location.

**Reasoning:** GeoPandas and shapely store points as `(x, y)`, which for geographic data means `(longitude, latitude)`, the reverse of the order in which places are often written. Swapping New York's `(-73.9, 40.7)` gives `(40.7, -73.9)`, and both numbers still pass range checks, because longitude allows -180 to 180 and latitude -90 to 90, yet the point lands in Antarctica. Only a semantic check, such as testing that the point lies inside the expected region's bounding box, detects this error, as the shared implementation for Exercises 3-4 shows.

### Exercise 5: answer contract

**Learner contract:** **Implementation:** Implement a longitude/latitude bounding-box validator with clear ordering and range checks. **Progressive hint:** Require west ≤ east, south ≤ north, and geographic bounds. **Verify:** Assert a valid box passes and separately reject west>east, south>north, longitude outside ±180, and latitude outside ±90.

**Reasoning:** Checking the bounds in `__post_init__` means an invalid box cannot be created at all, so every `GeographicBounds` object in the program is known to be valid. The chained comparisons `-180 <= west <= east <= 180` and `-90 <= south <= north <= 90` test order and range in one expression each, and `contains` is called with named arguments, so longitude and latitude cannot be swapped silently. The shared implementation for Exercises 5-7 checks a valid box and each of the four invalid kinds separately. This simple box cannot represent a region that crosses the 180-degree meridian, which needs a separate rule.

### Exercise 6: answer contract

**Learner contract:** **Debugging:** Repair a spatial join or overlay attempted before CRS comparison and reprojection. **Progressive hint:** Inspect `.crs`; transform one layer to the other's CRS. **Verify:** Show the pre-reprojection CRS mismatch, transform one layer, and assert the operation now uses equal CRS while preserving row identities.

**Reasoning:** A spatial join compares coordinates numerically, so joining points in degrees to regions in meters finds no matches: GeoPandas warns about the CRS mismatch but still returns a result in which every region is missing. The repair compares `.crs` first and transforms one layer with `to_crs`, so both layers use the same units before the join. The shared implementation for Exercises 5-7 reproduces the empty join, then asserts equal CRS, the expected regions, and that each store keeps its original row.

### Exercise 7: answer contract

**Learner contract:** **Edge case and explanation:** Define fallback behavior for missing/invalid geometry or unavailable map data, including an offline domain-graph alternative. **Progressive hint:** A missing basemap should not erase the analytical data layer. **Verify:** Feed missing/invalid geometry and unavailable basemap fixtures; assert analytical records remain represented by the documented local layer or graph fallback.

**Reasoning:** Missing and invalid geometries are records whose location is unknown or broken, not records to drop silently, so each gets a status, `valid`, `missing`, or `invalid`, and only valid shapes are drawn while the others are listed in a table. A basemap is context around the analytical layer, so when no cached basemap exists, the chart falls back to the local layer alone and says so in its title. The shared implementation for Exercises 5-7 checks that the drawn and listed records together account for every input record.

## Expanded mastery lab solutions

Treat coordinate reference system, coordinate order, source provenance, geometry validity, and network/cache behavior as part of the visualization.

### Shared implementation for Exercises 3-4: CRS and coordinate order

Longitude/latitude data are commonly stored as `(x, y) = (longitude, latitude)`.
Swapping them may still produce numbers inside valid ranges but a wrong place.
Degrees and projected meters cannot be overlaid without reprojection.


In [ ]:
import geopandas as gpd
from shapely.geometry import Point, box

cities = gpd.GeoDataFrame(
    {"name": ["San Francisco", "New York"]},
    geometry=[Point(-122.4, 37.8), Point(-73.9, 40.7)],  # (longitude, latitude)
    crs="EPSG:4326",
)
map_extent = gpd.GeoSeries([box(-125, 32, -70, 45)], crs="EPSG:4326").to_crs("EPSG:3857")
print(cities.crs.axis_info[0].unit_name, map_extent.crs.axis_info[0].unit_name)  # degree metre
print(cities.total_bounds.round(1), map_extent.total_bounds.round(0))
# Raw degrees sit next to the map origin, far outside the meter-based extent.
assert not map_extent.iloc[0].contains(box(*cities.total_bounds))
projected_cities = cities.to_crs(map_extent.crs)
assert projected_cities.crs == map_extent.crs
assert map_extent.iloc[0].contains(box(*projected_cities.total_bounds))

contiguous_us = box(-125, 24, -66, 50)  # west, south, east, north in degrees
new_york = Point(-73.9, 40.7)
swapped = Point(new_york.y, new_york.x)  # (40.7, -73.9)
assert -180 <= swapped.x <= 180 and -90 <= swapped.y <= 90  # range checks still pass
assert contiguous_us.contains(new_york)
assert not contiguous_us.contains(swapped)  # a point in Antarctica, not New York

### Shared implementation for Exercises 5-7: Validate spatial boundaries and degrade gracefully


In [ ]:
from dataclasses import dataclass


@dataclass(frozen=True)
class GeographicBounds:
    west: float
    south: float
    east: float
    north: float

    def __post_init__(self) -> None:
        if not (-180 <= self.west <= self.east <= 180):
            raise ValueError("longitude bounds must satisfy -180 <= west <= east <= 180")
        if not (-90 <= self.south <= self.north <= 90):
            raise ValueError("latitude bounds must satisfy -90 <= south <= north <= 90")

    def contains(self, longitude: float, latitude: float) -> bool:
        return self.west <= longitude <= self.east and self.south <= latitude <= self.north


bounds = GeographicBounds(west=-125, south=32, east=-114, north=42)
assert bounds.contains(longitude=-122.4, latitude=37.8)
assert not bounds.contains(longitude=37.8, latitude=-122.4)

invalid_boxes = {
    "west > east": (-114, 32, -125, 42),
    "south > north": (-125, 42, -114, 32),
    "longitude outside 180": (-190, 32, -114, 42),
    "latitude outside 90": (-125, -95, -114, 42),
}
for problem, (west, south, east, north) in invalid_boxes.items():
    try:
        GeographicBounds(west=west, south=south, east=east, north=north)
    except ValueError as error:
        print(f"{problem}: {error}")
    else:
        raise AssertionError(f"{problem} should be rejected")

# A spatial join before reprojection silently matches nothing.
import warnings
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import Point, Polygon, box

stores = gpd.GeoDataFrame(
    {"store": ["sf-01", "ny-07"]},
    geometry=[Point(-122.4, 37.8), Point(-73.9, 40.7)],
    crs="EPSG:4326",
)
regions = gpd.GeoDataFrame(
    {"region": ["west", "east"]},
    geometry=[box(-125, 32, -114, 42), box(-80, 38, -70, 45)],
    crs="EPSG:4326",
).to_crs("EPSG:3857")
assert stores.crs != regions.crs  # degrees versus meters
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    wrong = gpd.sjoin(stores, regions, how="left", predicate="within")
print(str(caught[0].message).splitlines()[0] if caught else "no warning")
assert wrong["region"].isna().all()
aligned = stores.to_crs(regions.crs)
joined = gpd.sjoin(aligned, regions, how="left", predicate="within")
assert aligned.crs == regions.crs
assert joined["store"].tolist() == ["sf-01", "ny-07"]  # row identities preserved
assert joined["region"].tolist() == ["west", "east"]

# Missing or invalid geometry stays visible as a record; a missing basemap is optional.
bowtie = Polygon([(0, 0), (1, 1), (1, 0), (0, 1), (0, 0)])  # self-intersecting
records = gpd.GeoDataFrame(
    {"site": ["a", "b", "c"]},
    geometry=[Point(-122.4, 37.8), None, bowtie],
    crs="EPSG:4326",
)
records["geometry_status"] = "valid"
records.loc[records.geometry.isna(), "geometry_status"] = "missing"
records.loc[records.geometry.notna() & ~records.geometry.is_valid, "geometry_status"] = "invalid"
drawable = records[records["geometry_status"].eq("valid")]
not_drawn = records.loc[records["geometry_status"].ne("valid"), ["site", "geometry_status"]]
assert len(drawable) + len(not_drawn) == len(records)  # every record is represented
print(not_drawn.to_dict("records"))

cached_basemap = Path("artifacts/day27/basemap.png")
basemap_note = "cached basemap" if cached_basemap.exists() else "no basemap: local layer only"
ax = drawable.plot()
ax.set_title(f"Sites ({basemap_note}; {len(not_drawn)} listed, not drawn)")
assert len(ax.collections) == 1
plt.close(ax.figure)

Before a GeoPandas operation, reject unknown CRSs and call `to_crs` so both
layers match. Filter/report empty or invalid geometries rather than silently
dropping them. If a cached basemap is unavailable, plot the local geometry
alone; if no geometry dependency is installed, a deterministic NetworkX graph
can teach domain topology without pretending it is a geographic map.
